In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import MessagePassing, global_mean_pool

from models_graph import HarmonicGraphEncoder
from models_FiLMatt import AttnFiLMSEModel

import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer

import pickle

import numpy as np

In [10]:
# load the dataset and the tokenizer
with open("data/gjt_train.pkl", "rb") as f:
    gjt_train = pickle.load(f)

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

graph_model = HarmonicGraphEncoder()
graph_model.to(device)

# load the model
model = AttnFiLMSEModel(
    chord_vocab_size=len(tokenizer.vocab),
    guidance_dim=graph_model.output_dim
)
model.to(device)

AttnFiLMSEModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (layers): ModuleList(
    (0-7): 8 x TransformerBlockWithAttnFiLM(
      (attn): MultiHeadAttentionWithAttnFiLM(
        (q_proj): Linear(in_features=512, out_features=512, bias=True)
        (k_proj): Linear(in_features=512, out_features=512, bias=True)
        (v_proj): Linear(in_features=512, out_features=512, bias=True)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
        (q_films): ModuleList(
          (0-7): 8 x IdCenteredFiLM(
            (gamma_proj): Linear(in_features=128, out_features=64, bias=True)
            (beta_proj): Linear(in_features=128, out_features=64, bias=True)
          )
        )
        (k_films): ModuleList(
          (0-7): 8 x IdCenteredFiLM(
            (gamma_proj): Linear(in_features=128, out_features=64, bias=True)
            (beta_proj): Linear(in_features=128, out_features=64, bias=True)
  

In [12]:
d = gjt_train[0]
print(d.keys())

dict_keys(['harmony_ids', 'attention_mask', 'pianoroll', 'time_signature', 'h_density_complexity', 'graph_ready_object'])


In [13]:
bar_start, bar_end = d['graph_ready_object'].get_valid_bar_segment_range(4)
print(bar_start, bar_end)

11 13


In [14]:
d['graph_ready_object'].make_graph_of_segment(bar_start, bar_end)
print(d['graph_ready_object'].segment_graph)

HeteroData(
  pitch={ x=[12, 12] },
  event={ x=[3, 1] },
  (pitch, participates, event)={
    edge_index=[2, 15],
    edge_attr=[15, 8],
  },
  (event, next, event)={
    edge_index=[2, 2],
    edge_attr=[2, 7],
  }
)


In [15]:
print(d['graph_ready_object'].segment_bar_objects)

[<graph_utils.Bar object at 0x7ac6f1c60c50>, <graph_utils.Bar object at 0x7ac6f1c61c10>]


In [16]:
token_positions = d['graph_ready_object'].get_token_positions_of_bar_segment()
print(token_positions)

[56, 57, 58, 59, 61, 62, 63, 64]
